In [38]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import faiss
import time

In [2]:
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6548.66it/s]


In [3]:
with open("500DaysofSummer.txt", "r", encoding="utf-8") as file:
    text = file.read()

In [4]:
def chunk_text(text, chunk_size=800, overlap=300):
    chunks = []

    for i in range(0, len(text), chunk_size - overlap):
        chunk = text[i:i + chunk_size]
        chunks.append(chunk)

    return chunks

In [5]:
chunks = chunk_text(text)
print(len(chunks))

205


In [6]:
emb = model.encode(
    chunks,
    convert_to_numpy=True,
    normalize_embeddings=True
)

emb = emb.astype("float32")

In [7]:
index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)

In [9]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

In [10]:
from groq import Groq

client = Groq(api_key=api_key)

In [31]:
query = "What city does the movie take place in?"
query_embedding = model.encode(
    query,
    convert_to_numpy=True,
    normalize_embeddings=True
)

query_embedding = query_embedding.reshape(1, -1).astype("float32")
retrieved_chunks = []
distances, indices = index.search(query_embedding, 10)
for idx in indices[0]:
    retrieved_chunks.append(chunks[idx])

context = "\n\n".join(retrieved_chunks)

distances, indices = index.search(query_embedding, 10)

In [32]:
prompt = f"""
You are a question-answering assistant.

Use ONLY the provided context.

Rules:

1. Never use outside knowledge.
2. Never infer information that is not explicitly stated.
3. If the answer is missing, reply exactly:
   "No information available in the provided context."
4. After every answer, include the exact sentence(s) from the context that support your answer.
5. If no supporting sentence exists, return only:
   "No information available in the provided context."

Context:
{context}

Question:
{query}

"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)
print("Question:" ,query)
print("Answer:" ,response.choices[0].message.content)

Question: What city does the movie take place in?
Answer: No information available in the provided context.


In [33]:
questions = [
    "Who is Tom?",
    "Who is Summer?",
    "Where does Tom work?",
    "What city does the story take place in?",
    "Who is Rachel?",
    "Why is Tom sad?",
    "What band do Tom and Summer like?",
    "What happens at IKEA?",
    "Who is Millie?",
    "How does the movie end?"
]

In [ ]:
for q in questions:
    print("="*80)
    print("Question:", q)
    # Retrieval
    query_embedding = model.encode([q]).astype("float32")
    distances, indices = index.search(query_embedding, k=12)

    # print("\nRetrieved Chunks:")
    # for rank, idx in enumerate(indices[0]):
    #     print(f"\n--- Chunk {idx} ---")
    #     print(chunks[idx][:500])

    context = "\n\n".join(chunks[idx] for idx in indices[0])

    prompt = f"""
You are a question-answering assistant.

Use ONLY the provided context.

Rules:

1. Never use outside knowledge.
2. Never infer information that is not explicitly stated.
3. If the answer is missing, reply exactly:
   "No information available in the provided context."
4. After every answer, include the exact sentence(s) from the context that support your answer.
5. If no supporting sentence exists, return only:
   "No information available in the provided context."

Context:
{context}

Question:
{q}

Answer:
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role":"user","content":prompt}]
    )

    print("\nAnswer:")
    print(response.choices[0].message.content)
    time.sleep(5)

Question: Who is Tom?

Answer:
Tom.

Supporting sentence: The first sentence of the provided context states: "t isn’t big, it is organized by a master. Two, on the walls, is a series of framed portraits, each one a famous building and its architectural blueprint." The context later refers to "Tom" in various places, clearly showing that "t" stands for Tom.

However this is confirmed in the following sentence where 'TOM' is used as a name: 
GIRL - Thomas.
TOM freezes.
Question: Who is Summer?

Answer:
Summer is a woman.

(Throughout the following, SUBTITLES will reveal specifics of the Narrator’s points.)
NARRATOR
Summer Finn was a woman.
 
FREEZE on SUMMER. (Throughout the following, SUBTITLES will
reveal specifics of the Narrator’s points.)
NARRATOR
Height: average.
Titles reveal specifics: 5’ 5”
NARRATOR
Weight: average.
Titles: 121 pounds.
NARRATOR
Shoe size: slightly above average.
Titles: Size 8.
NARRATOR
For all intents and purposes,
Summer Finn... just another girl.
Question: Wh